In [41]:
import re
from dotenv import load_dotenv
load_dotenv()

True

In [42]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple

class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [43]:
# Chunks
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)
with open("./report.md", "r") as f:
    text = f.read()
chunks = chunk_by_section(text)

In [44]:
# Embedding
import voyageai
vo = voyageai.Client()

def generate_embeddings(chunks, model="voyage-4-lite"):
    generate_embeddings = []
    for chunk in chunks:
        generate_embeddings.append(generate_embedding(chunk))
    return generate_embeddings

def generate_embedding(chunk, model="voyage-4-lite", input_type="query"):
    result = vo.embed(chunk, model=model, input_type=input_type)
    return result.embeddings[0]

In [45]:
generate_embeddings = generate_embeddings(chunks)

store = VectorIndex()
for embedding, chunk in zip(generate_embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})




[({'content': "Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts\n\nThe Legal department actively monitored and responded to several key developments this year. The ruling in _Synergy Dynamics v. Apex Solutions_ (Docket `CV-23-1101`) established a narrower interpretation of patent eligibility for certain software-implemented inventions, requiring a review of our current IP portfolio and filing strategy. Our team proactively identified potentially affected patents (Portfolio Segment ID: SW-PAT-CORE) and initiated amendments where necessary. Furthermore, ongoing efforts related to Project `GDPR-Audit-PhaseII` ensured continued compliance with evolving data privacy regulations, particularly concerning cross-border data transfers impacting research collaborations noted in Section 1 (Medical Research) and Section 9 (Pharmaceutical Development). A new internal framework (Policy Ref: `LEG-DP-FRMK-v3`) was implemented to streamline compliance processes. These legal

In [46]:
embedding = generate_embedding("What is `HRC/1921/DIP/045`?")
search = store.search(embedding, 2)
print(search)

[({'content': "Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts\n\nThe Legal department actively monitored and responded to several key developments this year. The ruling in _Synergy Dynamics v. Apex Solutions_ (Docket `CV-23-1101`) established a narrower interpretation of patent eligibility for certain software-implemented inventions, requiring a review of our current IP portfolio and filing strategy. Our team proactively identified potentially affected patents (Portfolio Segment ID: SW-PAT-CORE) and initiated amendments where necessary. Furthermore, ongoing efforts related to Project `GDPR-Audit-PhaseII` ensured continued compliance with evolving data privacy regulations, particularly concerning cross-border data transfers impacting research collaborations noted in Section 1 (Medical Research) and Section 9 (Pharmaceutical Development). A new internal framework (Policy Ref: `LEG-DP-FRMK-v3`) was implemented to streamline compliance processes. These legal